# Notebook 03 — Valores faltantes (NaN)

En los notebooks anteriores **ignoramos** que el dataset `penguins` tiene filas con datos faltantes. Hoy los enfrentamos. 🛠️

Trabajar con datos reales casi siempre implica lidiar con valores faltantes — encuestas incompletas, sensores que fallan, registros mezclados de distintas fuentes. Saber **detectarlos** y **decidir qué hacer con ellos** es una habilidad fundamental antes de hacer cualquier análisis o modelo.

## Objetivos de aprendizaje

1. Entender qué es **`NaN`** (Not a Number) y por qué aparece.
2. **Detectar** valores faltantes con `.isna()`, `.notna()` y `.isna().sum()`.
3. **Eliminar** filas con valores faltantes usando `.dropna()`.
4. **Rellenar** valores faltantes con `.fillna()` (con una constante o con un estadístico como la mediana).
5. Saber **cuándo elegir** cada estrategia.

---

## 1. Repaso rápido del Notebook 02

| Operación | Sintaxis |
|---|---|
| Filtrar por condición | `df[df["col"] > x]` |
| Combinar con AND / OR | `df[(c1) & (c2)]`, `df[(c1) | (c2)]` |
| Filtrar por lista | `df[df["col"].isin([...])]` |
| Filas + columnas | `df.loc[c1, ["col"]]` |
| Ordenar | `df.sort_values("col", ascending=False)` |

---

## 2. Setup

Cargamos `penguins` como siempre.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset("penguins")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

---

## 3. ¿Qué es `NaN`?

**`NaN`** significa *Not a Number*. En pandas representa **"valor faltante"** o **"dato no disponible"**. Lo verás en celdas que están vacías al cargar un CSV, en mediciones perdidas, o cuando una columna no aplica para una fila concreta.

Internamente pandas usa `numpy.nan`, que es un tipo especial de número de punto flotante.

### Cómo se ven los `NaN` en el dataset

Vamos a ver una fila que sí tiene valores faltantes:

In [ ]:
# Show rows that contain at least one NaN — observe how missing values appear
df[df.isna().any(axis=1)].head()

Las celdas con `NaN` se muestran literalmente como `NaN`. Estas filas no son un error: son datos que el dataset original no tenía y pandas los marca explícitamente para que tomes una decisión sobre ellos.

---

## 4. Detectar valores faltantes

Tres métodos clave:

| Método | Qué devuelve |
|---|---|
| `.isna()` | DataFrame del mismo tamaño con `True` donde hay NaN, `False` en otro caso |
| `.notna()` | Lo contrario: `True` donde **no** hay NaN |
| `.isna().sum()` | Cuenta de NaN **por columna** (porque `True` cuenta como 1) |

### Demo

In [ ]:
# A boolean DataFrame: True where the value is missing
df.isna().head()

In [ ]:
# Count of missing values per column
df.isna().sum()

In [ ]:
# Total number of rows with at least one missing value
df.isna().any(axis=1).sum()

### 🏋️ Ejercicio 4

Crea una `Series` llamada **`nan_count_per_column`** que contenga el número de valores faltantes en **cada columna** de `df`.

💡 Tip: `df.isna().sum()` ya devuelve exactamente eso.

In [ ]:
# YOUR CODE HERE
nan_count_per_column = None


In [ ]:
# Tests
assert isinstance(nan_count_per_column, pd.Series), "nan_count_per_column must be a pandas Series"
assert nan_count_per_column["sex"] == 11, f"Expected 11 NaN in 'sex', got {nan_count_per_column['sex']}"
assert nan_count_per_column["body_mass_g"] == 2, f"Expected 2 NaN in 'body_mass_g', got {nan_count_per_column['body_mass_g']}"
assert nan_count_per_column["species"] == 0, "'species' should have no missing values"
assert nan_count_per_column["island"] == 0, "'island' should have no missing values"

print("✅ ¡Bien! Detectaste correctamente los valores faltantes.")
print(nan_count_per_column)

---

## 5. Estrategia A — eliminar filas con `.dropna()`

La opción más simple: **borrar las filas que tengan valores faltantes**. Útil cuando son pocas y prefieres trabajar con datos completos.

### Variantes

```python
df.dropna()                    # elimina filas con cualquier NaN (default: how="any")
df.dropna(how="all")           # elimina solo filas que sean TODAS NaN
df.dropna(subset=["col"])      # elimina filas con NaN solo en columnas específicas
```

⚠️ **`.dropna()` no modifica el DataFrame original**: devuelve uno nuevo. Si quieres reemplazarlo, asígnalo a una variable.

### Demo

In [ ]:
# Drop any row with missing values
df_complete = df.dropna()
print(f"Original: {df.shape[0]} rows")
print(f"After dropna: {df_complete.shape[0]} rows")
print(f"Rows removed: {df.shape[0] - df_complete.shape[0]}")

In [ ]:
# Drop only rows where 'body_mass_g' is missing (keep rows where only 'sex' is missing)
df_with_mass = df.dropna(subset=["body_mass_g"])
print(f"After dropna(subset=['body_mass_g']): {df_with_mass.shape[0]} rows")

### 🏋️ Ejercicio 5

Crea un DataFrame llamado **`df_no_nan`** que contenga **solo las filas sin ningún valor faltante** del dataset original.

💡 Tip: `.dropna()` por defecto ya hace eso.

In [ ]:
# YOUR CODE HERE
df_no_nan = None


In [ ]:
# Tests
assert isinstance(df_no_nan, pd.DataFrame), "df_no_nan must be a DataFrame"
assert df_no_nan.shape[0] == 333, f"Expected 333 rows, got {df_no_nan.shape[0]}"
assert df_no_nan.isna().sum().sum() == 0, "df_no_nan must contain zero NaN values"
assert df_no_nan.shape[1] == df.shape[1], "Number of columns should not change"

print(f"✅ ¡Listo! Te quedaste con {df_no_nan.shape[0]} filas completas (perdiste {df.shape[0] - df_no_nan.shape[0]}).")

---

## 6. Estrategia B — rellenar con `.fillna()`

A veces no queremos perder filas: preferimos **imputar** (rellenar) los valores faltantes con algo razonable. `.fillna()` permite eso.

### Opciones comunes

```python
series.fillna(0)                  # constante numérica
series.fillna("Unknown")          # constante de texto
series.fillna(series.median())    # mediana de la columna
series.fillna(series.mean())      # media de la columna
series.fillna(series.mode()[0])   # moda (valor más frecuente)
```

Para columnas **numéricas** suele preferirse la **mediana** (más robusta a outliers que la media).
Para columnas **categóricas** se usa una etiqueta como `"Unknown"` o la **moda**.

### Demo

In [ ]:
# Fill the categorical 'sex' column with the string "Unknown"
sex_filled_demo = df["sex"].fillna("Unknown")
print("Value counts after filling:")
print(sex_filled_demo.value_counts())

In [ ]:
# Fill the numeric 'flipper_length_mm' column with its median
median_flipper = df["flipper_length_mm"].median()
print(f"Median flipper length: {median_flipper}")

flipper_filled_demo = df["flipper_length_mm"].fillna(median_flipper)
print(f"NaN after filling: {flipper_filled_demo.isna().sum()}")

### 🏋️ Ejercicio 6 — rellenar columna categórica

Crea una **`Series`** llamada **`sex_filled`** que sea la columna `"sex"` del DataFrame `df` con los valores `NaN` reemplazados por la cadena `"Unknown"`.

In [ ]:
# YOUR CODE HERE
sex_filled = None


In [ ]:
# Tests
assert isinstance(sex_filled, pd.Series), "sex_filled must be a pandas Series"
assert len(sex_filled) == len(df), "sex_filled must have the same length as df"
assert sex_filled.isna().sum() == 0, "sex_filled must have no NaN values"
assert (sex_filled == "Unknown").sum() == 11, "There should be exactly 11 'Unknown' values"
assert (sex_filled == "Male").sum() == 168, "Existing 'Male' values must be preserved"
assert (sex_filled == "Female").sum() == 165, "Existing 'Female' values must be preserved"

print("✅ ¡Genial! Rellenaste los 11 valores faltantes de 'sex' con 'Unknown'.")

### 🏋️ Ejercicio 7 — rellenar columna numérica con la mediana

Crea una **`Series`** llamada **`body_mass_filled`** que sea la columna `"body_mass_g"` del DataFrame `df` con los valores `NaN` reemplazados por la **mediana** de esa misma columna.

💡 Tip: calcula primero `df["body_mass_g"].median()`, luego úsalo en `.fillna(...)`.

In [ ]:
# YOUR CODE HERE
body_mass_filled = None


In [ ]:
# Tests
expected_median = df["body_mass_g"].median()
originally_missing_idx = df[df["body_mass_g"].isna()].index

assert isinstance(body_mass_filled, pd.Series), "body_mass_filled must be a pandas Series"
assert len(body_mass_filled) == len(df), "body_mass_filled must have the same length as df"
assert body_mass_filled.isna().sum() == 0, "body_mass_filled must have no NaN values"
assert (body_mass_filled.loc[originally_missing_idx] == expected_median).all(), f"Originally-missing rows must equal the median ({expected_median})"

print(f"✅ ¡Excelente! Rellenaste las 2 filas faltantes con la mediana ({expected_median} g).")

---

## 7. ¿Cuándo elegir cada estrategia?

No hay una regla universal, pero esta guía te servirá en el 90% de los casos:

| Situación | Estrategia recomendada |
|---|---|
| **Pocos NaN** (< 5% de las filas) y puedes permitirte perder filas | `.dropna()` |
| **Muchos NaN** y no quieres perder información | `.fillna()` con un estadístico (mediana / moda) |
| **NaN en una columna que no vas a usar** | Ignóralos: solo elimina/rellena al usarla |
| **NaN tienen significado** (ej. "no aplica") | Reemplaza con una etiqueta explícita (`"N/A"`, `0`, `-1`) |
| **Vas a entrenar un modelo** | Casi siempre necesitas tratar los NaN antes; muchos modelos no los aceptan |

### Reglas prácticas

1. **Siempre mira primero**: `df.isna().sum()` antes de decidir.
2. **Documenta tu decisión** (por ejemplo en un comentario): si rellenas con la mediana, todo análisis posterior asume eso.
3. **Nunca borres datos sin saber por qué faltan**. A veces los NaN son la pista más interesante del dataset.

---

## 8. Resumen — ¿qué aprendiste?

🎉 ¡Buen trabajo! Ya sabes manejar valores faltantes:

| Operación | Sintaxis |
|---|---|
| Boolean mask de NaN | `df.isna()` |
| Boolean mask de no-NaN | `df.notna()` |
| Conteo por columna | `df.isna().sum()` |
| Filas con algún NaN | `df[df.isna().any(axis=1)]` |
| Eliminar filas con NaN | `df.dropna()` |
| Eliminar solo si NaN en col | `df.dropna(subset=["col"])` |
| Rellenar con constante | `s.fillna("Unknown")` |
| Rellenar con mediana | `s.fillna(s.median())` |

## ¿Qué viene en el próximo notebook?

En **Notebook 04 — Agrupar y agregar (`groupby`)** vas a aprender a responder preguntas tipo:

- ¿Cuál es el peso **promedio** de cada especie de pingüino?
- ¿Cuántos pingüinos hay por **isla**?
- ¿Cómo varía el largo del pico **por especie y por sexo**?

El patrón **split-apply-combine** es uno de los conceptos más poderosos de pandas. ¡Nos vemos ahí! 📊